# 🧠 LSTM-Based Next Word Prediction
### Lab Assignment 5 — AI Sequence Prediction System

**Task:** Text Prediction — Predict the next word in a sentence  
**Model:** LSTM (Long Short-Term Memory) using TensorFlow/Keras  
**Dataset:** Custom small English text corpus (Shakespeare-style + general sentences)  

---
### 📚 What This Notebook Does:
1. Loads and cleans text data
2. Tokenizes text and creates sequences
3. Builds and trains an LSTM model
4. Predicts the next word from a given input
5. Saves the model and tokenizer for deployment

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run this in Google Colab)
!pip install tensorflow numpy pickle5 --quiet

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import pickle
import re

print("✅ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## 📂 Step 2: Load Dataset

**Dataset Used:** Custom English text corpus

We use a built-in text corpus from the NLTK Gutenberg dataset (free & open-source).  
Alternatively, you can use any `.txt` file.

**Dataset Source:** [Project Gutenberg](https://www.gutenberg.org/) via NLTK  
**Description:** Classic English literature text. We use a portion to keep training fast in Colab.  
**License:** Public Domain

In [ ]:
# Download NLTK corpus (free, public domain text)
import nltk
nltk.download('gutenberg', quiet=True)
nltk.download('punkt', quiet=True)

from nltk.corpus import gutenberg

# Use 'shakespeare-hamlet.txt' — a well-known public domain text
# Dataset link: https://www.nltk.org/book/ch02.html (NLTK Gutenberg corpus)
raw_text = gutenberg.raw('shakespeare-hamlet.txt')

# Show a sample of the text
print("📖 Sample text from dataset:")
print(raw_text[:500])
print(f"\n📊 Total characters in dataset: {len(raw_text)}")

## 🧹 Step 3: Data Preprocessing

### What we do here:
1. **Clean the text** — remove special characters, extra spaces, lowercase everything
2. **Tokenize** — convert each word to a number
3. **Create sequences** — sliding window approach (e.g., N words → predict next word)
4. **Pad sequences** — make all sequences the same length
5. **Create input-output pairs** — X = sequence of words, y = next word

In [ ]:
# -----------------------------------------------
# STEP 3A: Clean the Text
# -----------------------------------------------

def clean_text(text):
    """Clean raw text: lowercase, remove special characters, extra spaces."""
    text = text.lower()                          # Convert to lowercase
    text = re.sub(r'\[.*?\]', '', text)          # Remove text inside brackets
    text = re.sub(r'[^a-z\s]', '', text)         # Keep only letters and spaces
    text = re.sub(r'\s+', ' ', text)             # Remove extra whitespace
    text = text.strip()                          # Remove leading/trailing spaces
    return text

cleaned_text = clean_text(raw_text)

# Use only first 20,000 words to keep training fast in Colab
words = cleaned_text.split()
words = words[:20000]  # Limit for faster training
cleaned_text = ' '.join(words)

print("✅ Text cleaned!")
print(f"Total words (limited): {len(words)}")
print(f"Sample cleaned text: {cleaned_text[:200]}")

In [ ]:
# -----------------------------------------------
# STEP 3B: Tokenization
# Each unique word gets a unique integer ID
# Example: 'the' -> 1, 'king' -> 2, 'is' -> 3
# -----------------------------------------------

tokenizer = Tokenizer()
tokenizer.fit_on_texts([cleaned_text])  # Learn word->number mapping

# Total number of unique words in our vocabulary
total_words = len(tokenizer.word_index) + 1  # +1 because index starts at 1

print(f"✅ Tokenization complete!")
print(f"Total unique words (vocabulary size): {total_words}")
print(f"Sample word-to-index mapping: { {k: v for k, v in list(tokenizer.word_index.items())[:10]} }")

In [ ]:
# -----------------------------------------------
# STEP 3C: Create Input-Output Sequences
# 
# We use a sliding window approach:
# Example text: "to be or not to be"
# Sequence 1: [to]              → be
# Sequence 2: [to, be]          → or
# Sequence 3: [to, be, or]      → not
# etc.
# -----------------------------------------------

# Convert the full text into a list of token IDs
token_list = tokenizer.texts_to_sequences([cleaned_text])[0]
print(f"Total tokens in text: {len(token_list)}")

# Build n-gram sequences
input_sequences = []

# We use a window of 5 words (you can increase for better accuracy)
# For each position, take all words up to that point as a sequence
for i in range(1, len(token_list)):
    n_gram = token_list[max(0, i-10):i+1]  # Take up to 10 previous words + current
    input_sequences.append(n_gram)

# Limit to 15,000 sequences to avoid memory issues in Colab
input_sequences = input_sequences[:15000]

print(f"\n✅ Total sequences created: {len(input_sequences)}")
print(f"Sample sequence (token IDs): {input_sequences[5]}")

In [ ]:
# -----------------------------------------------
# STEP 3D: Padding Sequences
# 
# All sequences must be the SAME length for LSTM.
# Shorter sequences are padded with 0s at the beginning.
# Example: [0, 0, 3, 7, 12] (padded to length 5)
# -----------------------------------------------

# Find the length of the longest sequence
max_sequence_len = max([len(seq) for seq in input_sequences])
print(f"Maximum sequence length: {max_sequence_len}")

# Pad all sequences to the same length
input_sequences = np.array(pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding='pre'   # Pad with zeros at the BEGINNING
))

print(f"\n✅ Padding complete!")
print(f"Shape of padded sequences: {input_sequences.shape}")
print(f"Sample padded sequence: {input_sequences[5]}")

In [ ]:
# -----------------------------------------------
# STEP 3E: Split into Input (X) and Output (y)
# 
# X = all columns EXCEPT last → input words
# y = last column → the word to PREDICT
# 
# Example:
# Sequence: [0, 3, 7, 12, 45]
# X = [0, 3, 7, 12]   (input words)
# y = 45              (next word to predict)
# -----------------------------------------------

# X = all words except the last one
X = input_sequences[:, :-1]   # All columns except last

# y = the last word (what we want to predict)
y_labels = input_sequences[:, -1]  # Only last column

# Convert y to one-hot encoding
# Instead of y=45, we make y = [0,0,...,1,...,0] (a vector of 0s with 1 at position 45)
y = to_categorical(y_labels, num_classes=total_words)

print(f"✅ Input-Output pairs created!")
print(f"X shape (input sequences): {X.shape}")
print(f"y shape (output one-hot):  {y.shape}")
print(f"\nExample: X[0] = {X[0]}, y_label[0] = {y_labels[0]}")

## 🏗️ Step 4: Build LSTM Model

### Model Architecture:
```
Input Sequence
     ↓
Embedding Layer   ← Converts word IDs to dense vectors
     ↓
LSTM Layer 1      ← Learns patterns in sequences
     ↓
Dropout Layer     ← Prevents overfitting
     ↓
LSTM Layer 2      ← Deeper pattern learning
     ↓
Dense Layer       ← Fully connected
     ↓
Softmax Output    ← Probability for each word in vocabulary
```

In [ ]:
# -----------------------------------------------
# Build the LSTM Model
# -----------------------------------------------

def build_lstm_model(total_words, sequence_length):
    """Build and return the LSTM model."""
    
    model = Sequential([
        
        # Layer 1: Embedding Layer
        # Converts each word ID into a dense vector of size 100
        # Think of it as: each word gets a unique 100-dimension fingerprint
        Embedding(
            input_dim=total_words,     # Size of vocabulary
            output_dim=100,            # Size of each word vector
            input_length=sequence_length  # Length of input sequence
        ),
        
        # Layer 2: First LSTM Layer
        # return_sequences=True means output at EVERY timestep (needed for stacked LSTM)
        LSTM(150, return_sequences=True),
        
        # Layer 3: Dropout (randomly turns off 20% neurons to prevent overfitting)
        Dropout(0.2),
        
        # Layer 4: Second LSTM Layer
        # return_sequences=False means only output at LAST timestep
        LSTM(100),
        
        # Layer 5: Dropout again
        Dropout(0.2),
        
        # Layer 6: Dense Output Layer
        # Outputs probability for EVERY word in our vocabulary
        # softmax ensures all probabilities sum to 1.0
        Dense(total_words, activation='softmax')
    ])
    
    # Compile the model
    # categorical_crossentropy: loss function for multi-class classification
    # adam: a good default optimizer
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    
    return model

# Build the model
# X.shape[1] = length of each input sequence
model = build_lstm_model(total_words, X.shape[1])

# Print model summary
model.summary()

## 🏋️ Step 5: Train the Model

In [ ]:
# -----------------------------------------------
# Train the LSTM Model
# 
# epochs=30: Go through the entire dataset 30 times
# batch_size=64: Process 64 sequences at a time
# validation_split=0.1: Use 10% of data to check accuracy
# -----------------------------------------------

print("🚀 Starting model training...")
print("(This may take 5-15 minutes in Google Colab)")
print("-" * 50)

history = model.fit(
    X,                      # Input sequences
    y,                      # Output (next word, one-hot)
    epochs=30,              # Number of training rounds
    batch_size=64,          # Sequences per training step
    validation_split=0.1,   # Hold 10% for validation
    verbose=1               # Show progress
)

print("\n✅ Training complete!")

In [ ]:
# Plot training accuracy and loss
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss', color='red')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='green')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_plot.png', dpi=100)
plt.show()
print("\n✅ Training plot saved as 'training_plot.png'")

## 🔮 Step 6: Predict the Next Word

Now let's use our trained model to predict the next word given an input sentence.

In [ ]:
# -----------------------------------------------
# Prediction Function
# 
# Input:  A sentence like "to be or not"
# Output: The predicted next word, e.g., "to"
# -----------------------------------------------

def predict_next_word(model, tokenizer, text, max_sequence_len):
    """
    Given an input text, predict the next word.
    
    Parameters:
        model: trained LSTM model
        tokenizer: fitted tokenizer
        text: input sentence (string)
        max_sequence_len: the max sequence length used during training
    
    Returns:
        predicted_word: the next word as a string
    """
    # Step 1: Clean the input text
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 2: Convert input words to token IDs
    token_list = tokenizer.texts_to_sequences([text])[0]
    
    # Step 3: Pad the sequence to match training length
    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,  # -1 because model predicts position max_len
        padding='pre'
    )
    
    # Step 4: Get model predictions (probabilities for each word)
    predicted_probs = model.predict(token_list, verbose=0)
    
    # Step 5: Get the index of the word with highest probability
    predicted_index = np.argmax(predicted_probs, axis=1)[0]
    
    # Step 6: Convert index back to word
    predicted_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            predicted_word = word
            break
    
    return predicted_word

print("✅ Prediction function defined!")

In [ ]:
# -----------------------------------------------
# Test the Prediction
# -----------------------------------------------

# Test sentences — these should come from the training text domain
test_sentences = [
    "to be or not to",
    "the king of",
    "my lord",
    "what is",
    "good night sweet"
]

print("🔮 Next Word Predictions:")
print("-" * 45)

for sentence in test_sentences:
    next_word = predict_next_word(model, tokenizer, sentence, max_sequence_len)
    print(f"Input:  '{sentence}'")
    print(f"Predicted next word: '{next_word}'")
    print()

In [ ]:
# -----------------------------------------------
# Interactive Prediction
# Enter YOUR OWN sentence and get prediction!
# -----------------------------------------------

print("💬 Interactive Next Word Prediction")
print("Enter a sentence and the model will predict the next word!")
print("-" * 50)

user_input = input("Enter your sentence: ")
next_word = predict_next_word(model, tokenizer, user_input, max_sequence_len)

print(f"\n✅ Input: '{user_input}'")
print(f"🔮 Predicted next word: '{next_word}'")
print(f"📝 Complete: '{user_input} {next_word}'")  

## 💾 Step 7: Save Model and Tokenizer

In [ ]:
# -----------------------------------------------
# Save the trained model and tokenizer
# These files are used in FastAPI deployment
# -----------------------------------------------

# Save the model in .h5 format (Keras format)
model.save('lstm_model.h5')
print("✅ Model saved as 'lstm_model.h5'")

# Save the tokenizer using pickle
# We need the tokenizer in deployment to convert user input to token IDs
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("✅ Tokenizer saved as 'tokenizer.pkl'")

# Save max_sequence_len (needed during prediction in deployment)
with open('max_seq_len.pkl', 'wb') as f:
    pickle.dump(max_sequence_len, f)
print("✅ Max sequence length saved as 'max_seq_len.pkl'")

print("\n🎉 All files saved! Download them from the Colab file panel (left sidebar).")

In [ ]:
# -----------------------------------------------
# Download files from Google Colab to your PC
# -----------------------------------------------
from google.colab import files

files.download('lstm_model.h5')
files.download('tokenizer.pkl')
files.download('max_seq_len.pkl')
files.download('training_plot.png')

print("✅ Files downloaded! Use these in your FastAPI deployment.")

## 📊 Step 8: Model Summary & Results

In [ ]:
# Print final results summary
final_accuracy = history.history['accuracy'][-1]
final_val_accuracy = history.history['val_accuracy'][-1]
final_loss = history.history['loss'][-1]

print("=" * 50)
print("📊 TRAINING RESULTS SUMMARY")
print("=" * 50)
print(f"Dataset:          Shakespeare Hamlet (NLTK Gutenberg)")
print(f"Vocabulary size:  {total_words} unique words")
print(f"Training samples: {X.shape[0]} sequences")
print(f"Sequence length:  {X.shape[1]} tokens")
print(f"Epochs trained:   30")
print(f"Final Train Acc:  {final_accuracy:.4f} ({final_accuracy*100:.2f}%)")
print(f"Final Val Acc:    {final_val_accuracy:.4f} ({final_val_accuracy*100:.2f}%)")
print(f"Final Loss:       {final_loss:.4f}")
print("=" * 50)
print("\n✅ LSTM Model successfully trained and ready for deployment!")

---
## 🧠 LSTM Theory (Mandatory for Presentation)

### What is LSTM?
LSTM = Long Short-Term Memory. It's a special type of neural network that can **remember information over long sequences** — great for text and time-series!

### The 3 Gates of LSTM:

---

#### 1. 🔒 Forget Gate
**"What should I forget from memory?"**
- Looks at the previous hidden state `h(t-1)` and current input `x(t)`
- Outputs a value between 0 and 1 for each cell state element
- **0 = completely forget**, **1 = completely keep**
- Formula: `f(t) = sigmoid(Wf · [h(t-1), x(t)] + bf)`

---

#### 2. 📥 Input Gate
**"What new information should I store in memory?"**
- Two parts:
  - Input gate layer: decides which values to update
  - tanh layer: creates new candidate values to add
- Formula: `i(t) = sigmoid(Wi · [h(t-1), x(t)] + bi)`
- Candidate: `C̃(t) = tanh(Wc · [h(t-1), x(t)] + bc)`

---

#### 3. 📤 Output Gate
**"What should I output/pass forward?"**
- Decides what part of cell state to output as hidden state
- Formula: `o(t) = sigmoid(Wo · [h(t-1), x(t)] + bo)`
- Hidden state: `h(t) = o(t) * tanh(C(t))`

---

#### 4. 🧠 Cell State `C(t)`
- The **long-term memory** of LSTM
- Updated by: `C(t) = f(t) * C(t-1) + i(t) * C̃(t)`
- Old memory × forget + new info × input gate

#### 5. 💬 Hidden State `h(t)`
- The **short-term memory** / output at each timestep
- Passed to the next LSTM cell AND used for prediction
- `h(t) = o(t) * tanh(C(t))`

---

### Why LSTM for Text Prediction?
- Regular neural networks forget previous words
- LSTM **remembers context** over long sequences
- Example: In *"The king of Denmark was..."* — LSTM remembers *"king"* when predicting the next word after *"was"*